# Review by M2 — Data Preparation Module

**Module:** MATH1604 Modelling for Big Data  
**Role:** Team Member 2 — Download and Collation  
**File reviewed:** `scripts/data_preparation_M2.py`

## Purpose of this notebook

This notebook reviews and tests my M2 contribution to the group project.  
My task was to prepare the raw Python quiz response data by:

1. downloading the respondent answer files from the published cloud location;
2. saving them locally with consistent filenames;
3. collating the downloaded files into one combined file for later analysis.

The main output of this module is:

```text
output/collated_answers.txt
```


## Summary of the M2 requirements

The M2 module is responsible for two functions:

### 1. `download_answer_files(cloud_url, path_to_data_folder, total_respondents)`

The source files in the cloud dataset are named:

```text
a1.txt, a2.txt, ..., a64.txt
```

The function downloads these files from the cloud source and saves them locally using the required project naming format:

```text
answers_respondent_1.txt, answers_respondent_2.txt, ..., answers_respondent_64.txt
```

inside the `data/` folder.

For example:

```text
a1.txt  ->  answers_respondent_1.txt
a64.txt ->  answers_respondent_64.txt
```

### 2. `collate_answer_files(data_folder_path)`

This function combines the downloaded respondent files into:

```text
output/collated_answers.txt
```

Each respondent section is separated by a line containing a single asterisk:

```text
*
```


## Implementation overview

My implementation uses:

- `pathlib.Path` for safer cross-platform file path handling;
- input validation for the URL, local folder path, and number of respondents;
- `urllib.request.urlopen` to download the raw text files;
- exception handling for missing files, network errors, timeout errors, decoding errors, and file system errors;
- numeric sorting of respondent files before collation, so `answers_respondent_10.txt` is not placed before `answers_respondent_2.txt`.

The module does **not** change any respondent answers.  
If a respondent left questions unanswered, those unanswered questions are preserved exactly as they appear in the raw data.


## Test plan

The published dataset contains source files from `a1.txt` to `a64.txt`.

When these files are downloaded, my function saves them locally in the `data/` folder as:

```text
answers_respondent_1.txt, answers_respondent_2.txt, ..., answers_respondent_64.txt
```

To test robustness, I deliberately use:

```python
total_respondents = 70
```

This means the function attempts to download source files from `a1.txt` up to `a70.txt`.

The correct behaviour is:

- source files `a1.txt` to `a64.txt` are downloaded successfully;
- they are saved locally as `answers_respondent_1.txt` to `answers_respondent_64.txt`;
- source files `a65.txt` to `a70.txt` are skipped because they do not exist;
- no local files named `answers_respondent_65.txt` to `answers_respondent_70.txt` are created;
- the program continues running without crashing;
- the 64 downloaded files are collated into `output/collated_answers.txt`.

This test checks both normal functionality and missing-file handling.


In [17]:
from pathlib import Path
import sys

def find_project_root(start_path: Path) -> Path:
    """
    Find the project root by searching upwards until the scripts folder
    containing data_preparation_M2.py is found.
    """
    current = start_path.resolve()

    for folder in [current, *current.parents]:
        expected_file = folder / "scripts" / "data_preparation_M2.py"
        if expected_file.exists():
            return folder

    raise FileNotFoundError(
        "Could not find the project root. Make sure this notebook is inside "
        "the project repository and that scripts/data_preparation_M2.py exists."
    )

project_root = find_project_root(Path.cwd())

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print("Scripts folder exists:", (project_root / "scripts").exists())
print("M2 script exists:", (project_root / "scripts" / "data_preparation_M2.py").exists())


Project root: C:\Users\rokwi\Desktop\0\MATH1604_GP_G5-main\MATH1604_GP_G5-main
Scripts folder exists: True
M2 script exists: True


## Importing the M2 functions

The following cell imports the two functions from my M2 script.


In [18]:
from scripts.data_preparation_M2 import download_answer_files, collate_answer_files

print("Imported M2 functions successfully.")


Imported M2 functions successfully.


## Test 1 — Download the respondent files

This test downloads the respondent files from the published cloud source.

I deliberately request 70 source files even though only 64 are expected to exist.  
The correct behaviour is that the function downloads `a1.txt` to `a64.txt`, saves them as `answers_respondent_1.txt` to `answers_respondent_64.txt`, and skips the missing source files `a65.txt` to `a70.txt`.


In [19]:
cloud_url = "https://raw.githubusercontent.com/fc-leeds/MATH1604_2025_2026_data/main"
data_folder = project_root / "data"

downloaded_ids = download_answer_files(
    cloud_url=cloud_url,
    path_to_data_folder=str(data_folder),
    total_respondents=70
)

print("\nDownloaded file count:", len(downloaded_ids))
print("First downloaded respondent ID:", downloaded_ids[0] if downloaded_ids else None)
print("Last downloaded respondent ID:", downloaded_ids[-1] if downloaded_ids else None)
print("Downloaded respondent IDs:", downloaded_ids)


Downloaded a1.txt as answers_respondent_1.txt
Downloaded a2.txt as answers_respondent_2.txt
Downloaded a3.txt as answers_respondent_3.txt
Downloaded a4.txt as answers_respondent_4.txt
Downloaded a5.txt as answers_respondent_5.txt
Downloaded a6.txt as answers_respondent_6.txt
Downloaded a7.txt as answers_respondent_7.txt
Downloaded a8.txt as answers_respondent_8.txt
Downloaded a9.txt as answers_respondent_9.txt
Downloaded a10.txt as answers_respondent_10.txt
Downloaded a11.txt as answers_respondent_11.txt
Downloaded a12.txt as answers_respondent_12.txt
Downloaded a13.txt as answers_respondent_13.txt
Downloaded a14.txt as answers_respondent_14.txt
Downloaded a15.txt as answers_respondent_15.txt
Downloaded a16.txt as answers_respondent_16.txt
Downloaded a17.txt as answers_respondent_17.txt
Downloaded a18.txt as answers_respondent_18.txt
Downloaded a19.txt as answers_respondent_19.txt
Downloaded a20.txt as answers_respondent_20.txt
Downloaded a21.txt as answers_respondent_21.txt
Downloaded

### Expected result for Test 1

The expected result is:

```text
Downloaded file count: 64
First downloaded respondent ID: 1
Last downloaded respondent ID: 64
```

Files `a65.txt` to `a70.txt` should be reported as missing and skipped.  
This is not an error; it shows that the function handles unavailable source files safely.


## Test 2 — Check downloaded files

This cell checks that the expected local files exist in the `data/` folder.

The downloaded files should use the local naming format `answers_respondent_n.txt`, not the original source names `a1.txt`, `a2.txt`, and so on.


In [20]:
respondent_files = sorted(
    data_folder.glob("answers_respondent_*.txt"),
    key=lambda path: int(path.stem.split("_")[-1])
)

print("Number of respondent files in data folder:", len(respondent_files))
print("First local file:", respondent_files[0].name if respondent_files else None)
print("Last local file:", respondent_files[-1].name if respondent_files else None)

print("answers_respondent_1.txt exists:", (data_folder / "answers_respondent_1.txt").exists())
print("answers_respondent_64.txt exists:", (data_folder / "answers_respondent_64.txt").exists())
print("answers_respondent_65.txt exists:", (data_folder / "answers_respondent_65.txt").exists())
print("answers_respondent_70.txt exists:", (data_folder / "answers_respondent_70.txt").exists())


Number of respondent files in data folder: 64
First local file: answers_respondent_1.txt
Last local file: answers_respondent_64.txt
answers_respondent_1.txt exists: True
answers_respondent_64.txt exists: True
answers_respondent_65.txt exists: False
answers_respondent_70.txt exists: False


### Expected result for Test 2

The expected result is:

```text
Number of respondent files in data folder: 64
answers_respondent_1.txt exists: True
answers_respondent_64.txt exists: True
answers_respondent_65.txt exists: False
answers_respondent_70.txt exists: False
```

This confirms that the available source files were saved using the required local naming convention, and that no local files were created for missing source files.


## Test 3 — Collate the downloaded files

This test combines the downloaded local files into one output file.


In [21]:
collated_path = Path(collate_answer_files(str(data_folder)))

print("Collated file path:", collated_path)
print("Collated file exists:", collated_path.exists())



Collation summary
-----------------
Number of files collated: 64
First respondent file: answers_respondent_1.txt
Last respondent file: answers_respondent_64.txt
Collated file created at: C:\Users\rokwi\Desktop\0\MATH1604_GP_G5-main\MATH1604_GP_G5-main\output\collated_answers.txt
Collated file path: C:\Users\rokwi\Desktop\0\MATH1604_GP_G5-main\MATH1604_GP_G5-main\output\collated_answers.txt
Collated file exists: True


### Expected result for Test 3

The expected result is that this file exists:

```text
output/collated_answers.txt
```


## Test 4 — Validate the collated output file

This cell checks the structure of `collated_answers.txt`.

For 64 respondent files, I expect:

- 64 respondent sections;
- 63 separator lines containing `*`;
- 64 occurrences of `Question 1.`;
- 64 occurrences of `Question 100.`.


In [22]:
output_file = project_root / "output" / "collated_answers.txt"

text = output_file.read_text(encoding="utf-8")

sections = text.strip().split("\n*\n")

print("Output file exists:", output_file.exists())
print("Number of respondent sections:", len(sections))
print("Number of separator lines:", text.count("\n*\n"))
print("Number of Question 1 occurrences:", text.count("Question 1."))
print("Number of Question 100 occurrences:", text.count("Question 100."))


Output file exists: True
Number of respondent sections: 64
Number of separator lines: 63
Number of Question 1 occurrences: 64
Number of Question 100 occurrences: 64


### Expected result for Test 4

The expected result is:

```text
Number of respondent sections: 64
Number of separator lines: 63
Number of Question 1 occurrences: 64
Number of Question 100 occurrences: 64
```

This confirms that all 64 respondent files were collated and separated correctly.


## Test 5 — Check that each respondent section has 100 questions

This cell checks that every respondent section contains exactly 100 question blocks.


In [23]:
question_counts = []

for index, section in enumerate(sections, start=1):
    count = section.count("Question ")
    question_counts.append(count)

unique_question_counts = sorted(set(question_counts))

print("Unique question counts per respondent section:", unique_question_counts)
print("Minimum questions in a section:", min(question_counts))
print("Maximum questions in a section:", max(question_counts))

problem_sections = [
    index for index, count in enumerate(question_counts, start=1)
    if count != 100
]

print("Sections not containing 100 questions:", problem_sections)


Unique question counts per respondent section: [100]
Minimum questions in a section: 100
Maximum questions in a section: 100
Sections not containing 100 questions: []


### Expected result for Test 5

The expected result is:

```text
Unique question counts per respondent section: [100]
Sections not containing 100 questions: []
```

This confirms that no respondent section is missing questions after collation.


## Test 6 — Check selected and unanswered responses

The raw answer files can include unanswered questions.  
For a single question, the valid possibilities are:

- exactly one selected option, marked with `[X]`; or
- no selected option, meaning the question was unanswered.

This test checks that no question block has more than one selected option.


In [24]:
import re

question_pattern = re.compile(
    r"Question\s+\d+\..*?(?=\n\s*Question\s+\d+\.|\Z)",
    re.DOTALL
)

total_questions = 0
answered_questions = 0
unanswered_questions = 0
invalid_questions = []

for section_index, section in enumerate(sections, start=1):
    question_blocks = question_pattern.findall(section)

    for question_index, block in enumerate(question_blocks, start=1):
        total_questions += 1
        selected_count = block.count("[X]")

        if selected_count == 1:
            answered_questions += 1
        elif selected_count == 0:
            unanswered_questions += 1
        else:
            invalid_questions.append((section_index, question_index, selected_count))

print("Total question blocks:", total_questions)
print("Answered question blocks:", answered_questions)
print("Unanswered question blocks:", unanswered_questions)
print("Questions with more than one selected answer:", invalid_questions[:10])
print("Number of invalid question blocks:", len(invalid_questions))


Total question blocks: 6400
Answered question blocks: 6190
Unanswered question blocks: 210
Questions with more than one selected answer: []
Number of invalid question blocks: 0


### Interpretation of Test 6

The important result is:

```text
Number of invalid question blocks: 0
```

Unanswered questions are allowed because the raw dataset may contain questions where all four options are blank.  
The M2 module should preserve those unanswered questions instead of editing them.


## Evidence from testing

The M2 module successfully:

- downloaded the available source files `a1.txt` to `a64.txt`;
- skipped missing source files `a65.txt` to `a70.txt` without crashing;
- saved the downloaded files using the required local `answers_respondent_n.txt` naming format;
- avoided creating local files for missing respondents, such as `answers_respondent_65.txt`;
- collated the downloaded files into `output/collated_answers.txt`;
- separated respondent sections with asterisks;
- preserved unanswered questions without modifying the raw data.

The key evidence is that the final collated file contains 64 respondent sections and 63 separator lines.


## Limitations

One limitation is that downloading depends on internet access.  
If there is no internet connection, the download step cannot retrieve the raw source files.

However, once the respondent files have already been downloaded into the `data/` folder, the collation function can still run locally.


## Reflection

The strongest part of my M2 module is that it automates the preparation stage and produces a clean file structure for later analysis.  
The output files follow the required naming format, which makes the later modules easier to run.

The error handling is also important. When the function is asked to download source files beyond the published dataset, such as `a65.txt` to `a70.txt`, it does not stop the whole program. It reports the missing files and continues. This makes the module more robust.

The module also preserves the original responses. This is important because M2 should prepare and collate the data, not change the respondent answers.


## Conclusion

The M2 data preparation module works as intended.

It downloads the available raw respondent files, stores them using consistent filenames, and creates the required collated output file for the rest of the project pipeline.

The tests in this notebook provide evidence that the module is functional, robust, and suitable for use by the later analysis scripts.
